# LSTM Notebook

This notebook provides an example of working with the LSTM model. It
shows the process of loading per-game data for premier league forwards for
the current season, splitting into train/test sets, creating datasets and
dataloaders from them, training a `FootballLSTM` instance, and evaluating
it.

In [27]:
import sys
import os
import pandas as pd
from understatapi import UnderstatClient
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from matplotlib import pyplot as plt
import numpy as np

# Also import local helpers
target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)
    
from preprocess import get_position_players_stats_df, CustomFootballDataset
from football_lstm import FootballLSTM
from utils import hyperparam_tuning

In [2]:
# have GPU available to speed up
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cpu


In [3]:
# Create understat connection
understat = UnderstatClient()

In [4]:
# Choose what stats to forecast, and how many past games to use to predict
# the current one
stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
games_per_block = 10

In [5]:
### ONLY UNCOMMENT IF NEED TO PULL NEW POSITION DATA ###
### MAKE SURE THAT YOU CHANGE DF NAME ###

# Get forward stats for all seasons in top 5 leagues
# leagues = ["EPL", "La_Liga", "Bundesliga", "Serie_A", "Ligue_1"]
# seasons = [2021,2022,2023,2024,2025]
positions = ['F'] # forwards saved to csv, no need to run again
# 
# f_stats = get_position_players_stats_df(understat, positions, games_per_block, stats, leagues=leagues, seasons=seasons)
# f_stats.head()
# 
# # save dataset to csv to avoid pulls from understat every time as it can take 5-10min for just one position
# os.makedirs("data", exist_ok=True)  
# 
# # to be able to save datasets for a set of positions
# f_stats.to_csv(f"data/{'_'.join(positions)}_stats.csv", index=True)

In [6]:
# Read in saved df
f_stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats.csv").set_index(['player_id', 'date'])
f_stats_df.head()

### MAKE SURE THAT YOU CHANGE DF NAME IF NEW CSV ###
### AND THAT ALL DF BELOW ARE UPDATED TO MATCH NEW DF NAME ###

goals_per_90  xG_per_90  assists_per_90  xA_per_90  \
player_id date                                                             
3769      2017-11-05      0.176125   0.186441         0.35225   0.123798   
          2018-02-18      0.200445   0.136871         0.00000   0.056769   
          2019-08-25      0.000000   0.264276         0.00000   0.059867   
          2019-12-14      0.000000   0.061819         0.00000   0.191598   
          2020-07-04      0.000000   0.038386         0.00000   0.066023   

                      key_passes_per_90  xGChain_per_90  xGBuildup_per_90  
player_id date                                                             
3769      2017-11-05           2.113503        0.423613          0.130932  
          2018-02-18           1.603563        0.334553          0.154694  
          2019-08-25           1.022727        0.437595          0.128894  
          2019-12-14           1.152503        0.591171          0.429220  
          2020-07-04           0.978261        0.487978          0.383570

In [8]:
# Train/test split
train_len = int(len(f_stats_df) * 0.8)
train_df = f_stats_df.iloc[:train_len]
test_df = f_stats_df.iloc[train_len:]

In [9]:
train_df.head()

goals_per_90  xG_per_90  assists_per_90  xA_per_90  \
player_id date                                                             
3769      2017-11-05      0.176125   0.186441         0.35225   0.123798   
          2018-02-18      0.200445   0.136871         0.00000   0.056769   
          2019-08-25      0.000000   0.264276         0.00000   0.059867   
          2019-12-14      0.000000   0.061819         0.00000   0.191598   
          2020-07-04      0.000000   0.038386         0.00000   0.066023   

                      key_passes_per_90  xGChain_per_90  xGBuildup_per_90  
player_id date                                                             
3769      2017-11-05           2.113503        0.423613          0.130932  
          2018-02-18           1.603563        0.334553          0.154694  
          2019-08-25           1.022727        0.437595          0.128894  
          2019-12-14           1.152503        0.591171          0.429220  
          2020-07-04           0.978261        0.487978          0.383570

In [10]:
# Create datasets
blocks_per_input = 5

train_dataset = CustomFootballDataset(train_df, blocks_per_input, multiple_players=True)
test_dataset = CustomFootballDataset(test_df, blocks_per_input, multiple_players=True)

In [10]:
len(train_dataset)

8834

In [11]:
# Create dataloaders

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [12]:
# Tuning params
params = {
    "learning_rates": [0.1, 0.01, 0.001],      
    "epochs": [50, 75, 100],                  
    "h_sizes": [32, 64, 128],                  
    "layers": [1, 2, 3],                      
    "dropouts": [0.2, 0.3]                
}

opt_hyper_params = hyperparam_tuning(params, stats_df=f_stats_df, train_dataloader=train_dataloader, 
                                             test_dataloader=test_dataloader)

print(f"""
      Best Hyperparameter setup \n
      - learning rate: {opt_hyper_params['learning_rate']} 
      - number of epochs: {opt_hyper_params['epoch']}.
      - number of layers: {opt_hyper_params['layers']}.
      - hidden size: {opt_hyper_params['hidden_size']}.
      - dropout: {opt_hyper_params['dropout']}.
      """)


      Best Hyperparameter setup 

      - learning rate: 0.001 
      - number of epochs: 50.
      - number of layers: 1.
      - hidden size: 32.
      - dropout: 0.
      


In [13]:
# Tuned setup to MSE Loss and Adam
loss_fn = nn.MSELoss()
model = FootballLSTM(n_features=len(f_stats_df.columns), hidden_size=opt_hyper_params['hidden_size'], 
                     num_layers=opt_hyper_params['hidden_size'], dropout=opt_hyper_params['dropout']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=opt_hyper_params['learning_rate'])
num_epochs = opt_hyper_params['epoch']

# Train model
train_losses, test_losses = model.train_model(
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    n_epochs=num_epochs
)

In [14]:
# Evaluate test performance

rmse, mae = model.eval_model(test_dataloader)
print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mae}")

Test RMSE: 0.29844793677330017
Test MAE: 0.22876957058906555


In [1]:
# See how model does on forecasting a specific player--Erling Haaland

haaland_df = f_stats_df.loc[8260]

model.eval_model_on_player(haaland_df)

NameError: name 'f_stats_df' is not defined

In [45]:
# Get predictions for each player and save to csv

# For each player, get most recent blocks_per_input games, then predict next k
# with LSTM.

model = FootballLSTM(n_features=len(f_stats_df.columns), hidden_size=32, 
                     num_layers=1, dropout=0).to(device)

ids = []
preds = []

k = 20

for id, player_df in f_stats_df.groupby("player_id"):
    vals = player_df.values
    if len(vals) >= blocks_per_input:
        x = torch.tensor(vals[-blocks_per_input:], dtype=torch.float32).unsqueeze(0)
        player_preds = model.predict_next_k(x, k=k).squeeze(0).detach().cpu().numpy()
        preds.append(player_preds)
        ids.extend([id] * k)
        
preds = np.concatenate(preds)

In [49]:
# Turn into a df
preds_df = pd.DataFrame(preds)
# Add id column
preds_df.insert(0, 'player_id', ids)
preds_df.head()

,player_id,0,1,2,3,4,5,6
0,9,-0.038387,0.052088,0.032220,0.056791,-0.118244,-0.013727,-0.160009
1,9,-0.060066,0.071551,0.043712,0.036904,-0.081145,0.005084,-0.152594
2,9,-0.074671,0.071992,0.051258,0.029289,-0.062505,0.007115,-0.151723
3,9,-0.084200,0.069516,0.055586,0.025486,-0.054430,0.009399,-0.150354
4,9,-0.091725,0.063480,0.059463,0.024196,-0.050112,0.013034,-0.148258


In [48]:
# Add name column
id_names = pd.read_csv("../data/id_names.csv")
id_names["player_id"] = id_names["player_id"].astype("int64")

preds_df_names = pd.merge(preds_df, id_names, how="inner", on="player_id")
preds_df_names = preds_df_names[["player_id", "name", 0,1,2,3,4,5,6]]

preds_df_names.head()

,player_id,name,0,1,2,3,4,5,6
0,9,Tobias Kempe,-0.038387,0.052088,0.032220,0.056791,-0.118244,-0.013727,-0.160009
1,9,Tobias Kempe,-0.060066,0.071551,0.043712,0.036904,-0.081145,0.005084,-0.152594
2,9,Tobias Kempe,-0.074671,0.071992,0.051258,0.029289,-0.062505,0.007115,-0.151723
3,9,Tobias Kempe,-0.084200,0.069516,0.055586,0.025486,-0.054430,0.009399,-0.150354
4,9,Tobias Kempe,-0.091725,0.063480,0.059463,0.024196,-0.050112,0.013034,-0.148258


In [51]:
# Save to csv
preds_df_names.to_csv(f"../data/F_predictions_next_{k}.csv")